# Intrusão salina 1D — regime periódico e refinamento temporal

Este notebook continua a verificação com dados sintéticos inspirados no Rio São Mateus. O caso crítico usa $Q=2\,\mathrm{m^3/s}$ e a condição de Danckwerts em $x=L$.

A solução é avançada por ciclos completos de maré. Dois ciclos consecutivos são comparados sempre na mesma fase, e a execução só é interrompida quando três critérios ficam simultaneamente abaixo das tolerâncias por três ciclos sucessivos:

$$
\|C_k-C_{k-1}\|_\infty\leq 10^{-3}\ \mathrm{PSU},\qquad
\frac{|M_k-M_{k-1}|}{|M_{k-1}|}\leq 10^{-4},\qquad
|\overline{L}_{s,k}-\overline{L}_{s,k-1}|\leq 10\ \mathrm{m}.
$$

Aqui, $M_k$ representa o **conteúdo integrado de salinidade no domínio ao final do ciclo $k$**, definido por

$$
M_k=A\int_0^L C^{(k)}(x)\,dx,
$$

em que $A$ é a área da seção transversal e $C^{(k)}(x)$ é o perfil de salinidade ao final do ciclo. Como $C$ é expresso em PSU, $M_k$ deve ser interpretado como conteúdo integrado de salinidade ou estoque salino, e não rigorosamente como massa física de sal.

Neste experimento, $L=50\ \mathrm{km}$ é o comprimento do domínio computacional. Como $x=0$ representa uma seção efetiva situada a $10\ \mathrm{km}$ da foz, $x=L$ está a $60\ \mathrm{km}$ da foz. Assim, $L$ não é a distância absoluta medida desde a foz.

As forçantes têm magnitudes inspiradas no RSM, mas são funções periódicas idealizadas. Portanto, o experimento verifica o modelo numérico; não constitui calibração ou validação física do rio.

In [ ]:
# Execute esta célula uma vez, em uma sessão nova.
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    archive = Path("/content/salt_intrusion_1d_v0.4.0.zip")
    if not archive.exists():
        raise FileNotFoundError(
            "Envie salt_intrusion_1d_v0.4.0.zip para /content e execute novamente."
        )
    with zipfile.ZipFile(archive) as compressed:
        compressed.extractall("/content")
    project_dir = Path("/content/salt_intrusion_1d")
else:
    project_dir = Path("..").resolve()

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(project_dir)]
)
%matplotlib inline

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from salt_intrusion_1d import PeriodicityCriteria
from salt_intrusion_1d.experiment import MOUTH_OFFSET_KM
from salt_intrusion_1d.periodicity import (
    plot_boundary_temporal_refinement,
    plot_periodicity_diagnostics,
    run_periodicity_experiment,
    run_temporal_refinement,
    temporal_refinement_metrics,
    write_periodic_last_cycle_series,
    write_periodicity_diagnostics,
    write_temporal_refinement_summary,
)

## Execução automática

A busca pode alcançar algumas centenas de ciclos. O armazenamento completo do transiente é evitado: preservam-se os diagnósticos de cada ciclo e a série temporal completa apenas do último ciclo.

In [ ]:
criteria = PeriodicityCriteria(
    profile_linf_tolerance_psu=1.0e-3,
    relative_salt_tolerance=1.0e-4,
    mean_intrusion_tolerance_m=10.0,
    min_cycles=20,
    consecutive_cycles=3,
    max_cycles=400,
)

periodic = run_periodicity_experiment(
    discharge_m3_s=2.0,
    n_cells=500,
    dt_s=60.0,
    store_every_steps=30,
    criteria=criteria,
)

last = periodic.diagnostics[-1]
status = "atingido" if periodic.converged else "não atingido"
days = periodic.cycles_completed * periodic.config.tidal_period_s / 86_400
print(f"Regime periódico: {status}")
print(f"Ciclos simulados: {periodic.cycles_completed} ({days:.1f} dias)")
print(
    f"Intrusão média no último ciclo: "
    f"{last.mean_intrusion_m / 1_000 + MOUTH_OFFSET_KM:.3f} km da foz"
)
print(
    f"Intrusão máxima no último ciclo: "
    f"{last.max_intrusion_m / 1_000 + MOUTH_OFFSET_KM:.3f} km da foz"
)

## Evolução ciclo a ciclo

In [ ]:
cycles = np.array([item.cycle for item in periodic.diagnostics])
mean_intrusion = np.array(
    [item.mean_intrusion_m for item in periodic.diagnostics]
) / 1_000 + MOUTH_OFFSET_KM

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(cycles, mean_intrusion, linewidth=2.0)
ax.set_xlabel("Ciclo de maré")
ax.set_ylabel("Intrusão média a partir da foz (km)")
ax.set_title("Evolução até o regime periódico")
ax.grid(alpha=0.25)
plt.show()

## Último ciclo de maré

In [ ]:
cycle = periodic.last_cycle
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(
    cycle.times_s / 3_600,
    cycle.intrusion_length_m / 1_000 + MOUTH_OFFSET_KM,
    linewidth=2.0,
)
ax.set_xlabel("Tempo no último ciclo de maré (h)")
ax.set_ylabel("Distância da frente salina à foz (km)")
ax.set_title(f"Último ciclo simulado (ciclo {periodic.cycles_completed})")
ax.grid(alpha=0.25)
plt.show()

## Investigação do máximo de $C(L,t)$

O máximo pontudo observado em $C(L,t)$ é comparado aos instantes exatos em que $v(t)$ muda de sinal. A inversão de maior interesse é $v>0\rightarrow v<0$, pois nesse instante $x=L$ deixa de ser uma fronteira de saída com Neumann homogênea e passa a ser uma entrada com a condição de Danckwerts.

Para avaliar a influência da discretização temporal, a solução periódica é recalculada com

$$
\Delta t_{\mathrm{ref}}=\frac{\Delta t}{2}=30\ \mathrm{s},
$$

mantendo a malha espacial, as forçantes físicas e os critérios de periodicidade.

In [ ]:
temporal_results = run_temporal_refinement(
    periodic,
    refinement_factor=2,
)
metrics = temporal_refinement_metrics(temporal_results)

print(
    f"Inversão v>0 para v<0: "
    f"{metrics['velocity_reversal_time_h']:.5f} h"
)
print(
    f"Máximo com Δt={metrics['coarse_dt_s']:.0f} s: "
    f"{metrics['coarse_peak_salinity_psu']:.5f} PSU em "
    f"{metrics['coarse_peak_time_h']:.5f} h "
    f"({metrics['coarse_peak_minus_reversal_s']:+.2f} s da inversão)"
)
print(
    f"Máximo com Δt={metrics['refined_dt_s']:.0f} s: "
    f"{metrics['refined_peak_salinity_psu']:.5f} PSU em "
    f"{metrics['refined_peak_time_h']:.5f} h "
    f"({metrics['refined_peak_minus_reversal_s']:+.2f} s da inversão)"
)
print(
    f"Mudança na intrusão média: "
    f"{metrics['mean_intrusion_change_m']:.2f} m"
)
print(
    f"Mudança na intrusão máxima: "
    f"{metrics['max_intrusion_change_m']:.2f} m"
)

fig = plot_boundary_temporal_refinement(temporal_results)
plt.show()

### Interpretação

Na configuração padrão, o máximo de $C(L,t)$ fica a menos de um passo temporal da inversão $v>0\rightarrow v<0$. Ao reduzir $\Delta t$ de 60 s para 30 s, o formato pontudo permanece e a altura do máximo varia menos de $0{,}005$ PSU. Portanto, o máximo está associado à troca Neumann–Danckwerts; o refinamento revela apenas uma pequena dependência quantitativa esperada para o método temporal de primeira ordem.

## Exportação

In [ ]:
output_dir = (
    Path("/content/results_periodicity")
    if IN_COLAB
    else Path("../results_periodicity")
)
output_dir.mkdir(parents=True, exist_ok=True)

write_periodicity_diagnostics(periodic, output_dir)
write_periodic_last_cycle_series(periodic, output_dir)
plot_periodicity_diagnostics(periodic, output_dir)
write_temporal_refinement_summary(temporal_results, output_dir)
fig = plot_boundary_temporal_refinement(
    temporal_results,
    output_dir / "temporal_refinement_right_boundary.png",
)
plt.close(fig)

print(f"Arquivos salvos em: {output_dir.resolve()}")